In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../train.csv")

X = df.drop(columns=["Price_INR_Lakhs", "Property_ID"])
y = df["Price_INR_Lakhs"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42
)

categorical_features = [
    "City", "Locality_Type", "Property_Type", "Furnishing_Status"
]

numerical_features = [
    "BHK", "Bathrooms", "Super_Area_SqFt", "Carpet_Area_SqFt",
    "Floor_Number", "Total_Floors", "Age_of_Property",
    "Parking", "Lift_Available", "Gated_Community",
    "Distance_to_Metro_km", "Distance_to_City_Center_km"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)],
    sparse_threshold=0
)

### Model comparison on the same validation split

All models below use the same 80/20 split (`random_state=42`) as notebooks 20 to 29.

In [2]:
linear_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

models = {
    "Linear Regression": Pipeline([
        ("preprocessor", linear_preprocessor),
        ("regressor", LinearRegression())
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
    "HistGB baseline": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", HistGradientBoostingRegressor(
            learning_rate=0.10, max_iter=100, max_leaf_nodes=31, random_state=42
        ))
    ]),
    "HistGB tuned": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", HistGradientBoostingRegressor(
            learning_rate=0.05, max_iter=500, max_leaf_nodes=15,
            min_samples_leaf=100, l2_regularization=1.0, random_state=42
        ))
    ])
}

In [3]:
import time

comparison = []

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_val)

    comparison.append({
        "model": name,
        "MAE": mean_absolute_error(y_val, y_pred),
        "RMSE": mean_squared_error(y_val, y_pred) ** 0.5,
        "R2": r2_score(y_val, y_pred),
        "train_time_s": train_time
    })

comparison_df = pd.DataFrame(comparison).sort_values("R2", ascending=False)
comparison_df.round(4)

,model,MAE,RMSE,R2,train_time_s
3,HistGB tuned,32.5842,68.0629,0.7908,0.9553
2,HistGB baseline,32.8465,68.6725,0.7870,0.6094
1,Random Forest,34.0516,70.4281,0.7760,10.5869
0,Linear Regression,47.9403,86.0255,0.6657,0.4143


The tuned HistGradientBoosting model is the best so far (R2 0.7908) and trains about 10x faster than
the Random Forest. Linear Regression is far behind, which shows the price relationship is strongly non-linear.